# 常驻AI CPU自检

先停止主程序。在本目录运行下面代码，检查三张记录图片的四角和常驻进程。成功后再运行 p11_start.ipynb。


In [ ]:
from pathlib import Path
import os,signal,subprocess,sys
ROOT=Path.cwd().resolve()
if not (ROOT/'v99_ai_selftest.py').is_file():
    raise RuntimeError('请在新的常驻AI运行包目录打开这个Notebook。')
print('开始CPU自检：只用三张记录图片，不访问摄像头或FPGA。',flush=True)
child=subprocess.Popen([sys.executable,'-u','-B',str(ROOT/'v99_ai_selftest.py')],cwd=str(ROOT),
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,
    env=dict(os.environ,PYTHONUNBUFFERED='1'))
try:
    for line in child.stdout:print(line,end='',flush=True)
    code=child.wait()
except KeyboardInterrupt:
    child.send_signal(signal.SIGINT)
    try:child.wait(4)
    except subprocess.TimeoutExpired:child.kill();child.wait()
    raise
if code:
    raise RuntimeError('自检没有通过，请提供本目录 ai_selftest_result.json。')
print('3/3四角检查通过。现在可以运行 p11_start.ipynb；待主页AI就绪后进入实时。')
